# Génération de format de chargement de tarifs

Génération de deux fichiers parquet pour les tarifs et pour les liens tarif-pdc.

In [1]:
from datetime import datetime, timezone
import sys
from pathlib import Path
import json
import pandas as pd
# remonte à la racine du projet OCPI
sys.path.append(str(Path().resolve().parents[0]))
from source.tariff import TariffObject, TariffDimensionTypeEnum, TariffElement, TariffElements, PriceComponent

In [2]:
statiques = pd.read_csv("../data/qualicharge/donnees_statiques___liste_stations_pdcs_puissance_2026-06-06T23_40.csv", sep=",")
total_50 = len(statiques[statiques["puissance_nominale"] >= 50])
total_50

37157

In [3]:
def tariffs_to_dataframe(tariffs: list[dict], schema: dict, verbose: bool = False) -> pd.DataFrame:
    original_id = []
    original_last_updated = []
    raw = []
    start= []
    end = []
    for tarif in tariffs:
        assert TariffObject.is_valid_json(schema, tarif, verbose=verbose), f"Tariff {tarif['id']} is not valid according to the schema"
        original_id.append(tarif["country_code"] + tarif["party_id"] + tarif["id"])
        original_last_updated.append(tarif["last_updated"])
        raw.append(tarif)
        start.append(max(datetime.fromisoformat(tarif["last_updated"]), datetime.fromisoformat(tarif.get("start_date_time", datetime(1970, 1, 1, tzinfo=timezone.utc).isoformat()))))
        end.append(datetime.fromisoformat(tarif["end_date_time"]) if tarif.get("end_date_time") else None)
    return pd.DataFrame({
        "original_id": original_id,
        "original_last_updated": original_last_updated,
        "raw": raw,
        "start": start,
        "end": end})

In [4]:
def tariffs_from_OCPI(locations: list, tariffs: list, schema: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    tariff = tariffs_to_dataframe(tariffs, schema)  
    id_pdc_itinerance = []
    id_tariff = []
    for location in locations:
        for evse in location["evses"]:
            id_pdc_itinerance.append(evse["evse_id"].replace("*", ""))
            tariff_id = location["country_code"] + location["party_id"]
            tariff_id+= evse["connectors"][0]["tariff_ids"][0]
            if tariff_id not in tariff["original_id"].values:
                print(f"Tariff ID {tariff_id} not found in original tariffs")
            id_tariff.append(tariff_id)
    tariffpdc = pd.DataFrame({
        "id_pdc_itinerance": id_pdc_itinerance,
        "id_tariff": id_tariff})
    return (tariff, tariffpdc)

In [5]:
def tariff_simple(country_code: str, party_id: str, id: str, last_updated: str, dimension: str, price: float, start_date_time: str=None, end_date_time: str=None):
    price_component = PriceComponent(type=TariffDimensionTypeEnum(dimension), price=price, step_size=1)
    tariff_elements = TariffElements(
        root=[TariffElement(price_components=[price_component])]
    )
    tariff = TariffObject(
        country_code=country_code,
        party_id=party_id,
        id=id,
        elements=tariff_elements,
        last_updated=last_updated,
        start_date_time=start_date_time,
        end_date_time=end_date_time
    )
    return json.loads(tariff.model_dump_json(exclude_none=True))

## Tesla

In [6]:
with open('../data/tesla/FR_Supercharging_locations.json', 'r', encoding='utf-8') as f:
    tesla_locations = json.load(f)
with open('../data/tesla/FR_Supercharging_NTSLA_tariffs.json', 'r', encoding='utf-8') as f:
    tesla_tariffs = json.load(f)
len(tesla_locations["data"]), len(tesla_tariffs["data"])
with open("../source/schema.json") as f:
    schema = json.load(f)
tariffs, tariffpdc = tariffs_from_OCPI(tesla_locations["data"], tesla_tariffs["data"], schema)
total_tesla = len(tariffpdc)
print(f"len tariff, tariffpdc et % : {len(tariffs)}, {total_tesla}, {total_tesla/total_50*100:.2f} %")

tariffs.to_parquet("../data/tesla/qualicharge_tariff.parquet", index=False)
tariffpdc.to_parquet("../data/tesla/qualicharge_tariffpdc.parquet", index=False)
tariffpdc

len tariff, tariffpdc et % : 262, 3898, 10.49 %


,id_pdc_itinerance,id_tariff
0,FRTSLEA4QY2C,NLTSL5fa1c37d-9aa9-4778-805b-8a1c1299765d
1,FRTSLEA4T0JE,NLTSL5fa1c37d-9aa9-4778-805b-8a1c1299765d
2,FRTSLEA6K9GP,NLTSL5fa1c37d-9aa9-4778-805b-8a1c1299765d
3,FRTSLEA6LMC5,NLTSL5fa1c37d-9aa9-4778-805b-8a1c1299765d
4,FRTSLE0003DB,NLTSL80791b33-aedd-4d7d-a0b8-1d831c44de68
...,...,...
3893,FRTSLECM5I9O,NLTSL64395007-0c2e-45bf-a948-1148c3a15815
3894,FRTSLECM5KYZ,NLTSL64395007-0c2e-45bf-a948-1148c3a15815
3895,FRTSLECM5QVJ,NLTSL64395007-0c2e-45bf-a948-1148c3a15815
3896,FRTSLECM5QVG,NLTSL64395007-0c2e-45bf-a948-1148c3a15815


In [7]:
with open("../data/tesla/FR_Supercharging_NTSLA_tariffs.json") as f:
    examples_data = json.load(f)
text = "# Tarifs au format texte des tarifs Tesla" + "\n"
for example in examples_data["data"]:
    tariff = TariffObject.model_validate(example)
    text += tariff.to_text() + "\n"
with open("../data/tesla/tariffs_text.md", "w", encoding="utf-8") as f:
    f.write(text)

## Fastned

In [8]:
with open("../source/schema.json") as f:
    schema = json.load(f)
tariff = tariff_simple(
    country_code="FR",
    party_id="FAS",
    id="tarif_unique_1",
    last_updated="2024-06-01T12:00:00Z",
    dimension="ENERGY",
    price=0.5083) # 0.61€/kWh TTC -> 0.61/1.2 = 0.5083€/kWh HT)
tariffs = tariffs_to_dataframe([tariff], schema, verbose=False)
#print(tariffs)
statiques_fastned = statiques[statiques["id_pdc_itinerance"].str.startswith("FRFAS", na=False)]
tariffpdc = pd.DataFrame({
        "id_pdc_itinerance": statiques_fastned["id_pdc_itinerance"].tolist(),
        "id_tariff": ["FRFAStarif_unique_1"] * len(statiques_fastned)})
total_fastned = len(tariffpdc)
print(f"len tariff, tariffpdc et % : {len(tariffs)}, {total_fastned}, {total_fastned/total_50*100:.2f} %")
tariffs.to_parquet("../data/fastned/qualicharge_tariff.parquet", index=False)
tariffpdc.to_parquet("../data/fastned/qualicharge_tariffpdc.parquet", index=False)
tariffpdc


len tariff, tariffpdc et % : 1, 457, 1.23 %


,id_pdc_itinerance,id_tariff
0,FRFASE3323310,FRFAStarif_unique_1
1,FRFASE3327905,FRFAStarif_unique_1
2,FRFASE3310001,FRFAStarif_unique_1
3,FRFASE3322903,FRFAStarif_unique_1
4,FRFASE3309605,FRFAStarif_unique_1
...,...,...
452,FRFASE3300801,FRFAStarif_unique_1
453,FRFASE3311003,FRFAStarif_unique_1
454,FRFASE3332104,FRFAStarif_unique_1
455,FRFASE3302702,FRFAStarif_unique_1


## Total

In [9]:

with open('../data/total/Tarifs.json', 'r', encoding='utf-8') as f:
    total_tarifs = json.load(f)
len(total_tarifs)

65

## Driveco

In [10]:

with open('../data/driveco/Tarifs.json', 'r', encoding='utf-8') as f:
    driveco_tarifs = json.load(f)
len(driveco_tarifs)

85